In [ ]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="YOUR_API_KEY")
project = rf.workspace("dharini-y4why").project("violence-gedzq")
version = project.version(5)
dataset = version.download("yolo26")

In [ ]:
import os

for root, dirs, files in os.walk(dataset.location):
    print(root)
    break

In [ ]:
labels_path = os.path.join(dataset.location, "train", "labels")
print("Archivos label:", os.listdir(labels_path)[:5])

In [ ]:
sample_label = os.listdir(labels_path)[0]
with open(os.path.join(labels_path, sample_label), "r") as f:
    print(f.read())

In [ ]:
import shutil
from pathlib import Path
import yaml

# Paths
SRC_DATASET = Path(dataset.location)
OUT_DATASET = SRC_DATASET.parent / f"{SRC_DATASET.name}_2cls"

# Class grouping
VIOLENCE_NAMES = {"Aggression", "fighting"}
WEAPON_NAMES   = {"Holding a bottle", "Holding aGun", "Holding astick"}

# Load original YAML
with open(SRC_DATASET / "data.yaml", "r") as f:
    data_yaml = yaml.safe_load(f)

original_names = data_yaml["names"]

# Build class id mapping
oldid_to_newid = {}
for i, name in enumerate(original_names):
    if name in VIOLENCE_NAMES:
        oldid_to_newid[i] = 0
    elif name in WEAPON_NAMES:
        oldid_to_newid[i] = 1

print("Mapping:", oldid_to_newid)

# Recreate output directory
if OUT_DATASET.exists():
    shutil.rmtree(OUT_DATASET)
OUT_DATASET.mkdir(parents=True)

# Process each split
for split in ["train", "valid", "test"]:
    img_src = SRC_DATASET / split / "images"
    lbl_src = SRC_DATASET / split / "labels"

    img_dst = OUT_DATASET / split / "images"
    lbl_dst = OUT_DATASET / split / "labels"

    img_dst.mkdir(parents=True, exist_ok=True)
    lbl_dst.mkdir(parents=True, exist_ok=True)

    for img_path in img_src.glob("*.*"):
        shutil.copy(img_path, img_dst / img_path.name)

        label_path = lbl_src / (img_path.stem + ".txt")

        if label_path.exists():
            new_lines = []
            for line in label_path.read_text().splitlines():
                parts = line.split()
                old_id = int(parts[0])

                if old_id in oldid_to_newid:
                    parts[0] = str(oldid_to_newid[old_id])
                    new_lines.append(" ".join(parts))

            (lbl_dst / label_path.name).write_text("\n".join(new_lines))
        else:
            (lbl_dst / (img_path.stem + ".txt")).write_text("")

# Create new YAML
new_yaml = {
    "path": str(OUT_DATASET),
    "train": "train/images",
    "val": "valid/images",
    "test": "test/images",
    "names": ["violence", "weapon"]
}

with open(OUT_DATASET / "data.yaml", "w") as f:
    yaml.safe_dump(new_yaml, f, sort_keys=False)

print("2-class dataset created at:", OUT_DATASET)

In [ ]:
from collections import Counter
import glob

label_files = glob.glob(f"{OUT_DATASET}/train/labels/*.txt")

counter = Counter()

for file in label_files:
    for line in open(file):
        cls = int(line.split()[0])
        counter[cls] += 1

print(counter)

In [ ]:
!pip install ultralytics
!yolo detect train \
data={OUT_DATASET}/data.yaml \
model=yolo26n.pt \
epochs=50 \
imgsz=640 \
batch=16

In [ ]:
!pip install -q kaggle

In [ ]:
import os
import kagglehub
os.environ['KAGGLE_USERNAME'] = "YOURS"
os.environ['KAGGLE_KEY'] = "YOURS"


path = kagglehub.dataset_download("mohamedmustafa/real-life-violence-situations-dataset")


print("El dataset se descargó en:", path)

In [ ]:
import shutil
import os


destino = "/content/dataset_videos"

shutil.copytree(path, destino, dirs_exist_ok=True)

print(f"¡Listo! Ahora puedes ver tus carpetas 'Violence' y 'NonViolence' en la carpeta: {destino}")

In [ ]:
import os
import shutil
import random
from pathlib import Path

# Configuración de rutas CORREGIDA
# Agregamos la subcarpeta intermedia que venía en el zip
dataset_path = Path('/content/dataset_videos/Real Life Violence Dataset')
output_path = Path('/content/rwf_split')

# Proporción de entrenamiento (0.8 = 80%)
split_ratio = 0.8

# Crear estructura de carpetas
for split in ['train', 'val']:
    for cls in ['fight', 'no_fight']:
        (output_path / split / cls).mkdir(parents=True, exist_ok=True)

# Mapeo de carpetas originales a nuevas clases
mapping = {
    'Violence': 'fight',
    'NonViolence': 'no_fight'
}

for folder_name, class_name in mapping.items():
    src_folder = dataset_path / folder_name

    # Validamos que la carpeta exista por seguridad
    if not src_folder.exists():
        print(f"Error: No se encontró la carpeta {src_folder}")
        continue

    videos = os.listdir(src_folder)
    random.shuffle(videos)

    # Calcular punto de corte
    split_idx = int(len(videos) * split_ratio)
    train_vids = videos[:split_idx]
    val_vids = videos[split_idx:]

    # Copiar archivos a train
    for vid in train_vids:
        shutil.copy(src_folder / vid, output_path / 'train' / class_name / vid)

    # Copiar archivos a val
    for vid in val_vids:
        shutil.copy(src_folder / vid, output_path / 'val' / class_name / vid)

print(f"¡Split completado! Los archivos están en: {output_path}")
print(f"Train: {len(os.listdir(output_path/'train'/'fight'))} videos de pelea")
print(f"Val: {len(os.listdir(output_path/'val'/'fight'))} videos de pelea")

In [ ]:
from pathlib import Path
import os

# Apuntamos a la carpeta que ya organizamos y dividimos en la celda 28
OUT_VIDEOS = Path("/content/rwf_split")

# Definimos las rutas de entrenamiento (Train)
TRAIN_FIGHT_DIR = OUT_VIDEOS / "train" / "fight"
TRAIN_NOFIGHT_DIR = OUT_VIDEOS / "train" / "no_fight"

# Definimos las rutas de validación (Val)
VAL_FIGHT_DIR = OUT_VIDEOS / "val" / "fight"
VAL_NOFIGHT_DIR = OUT_VIDEOS / "val" / "no_fight"

print("¡Directorios listos y enlazados con éxito!")
print("-" * 30)
print(f"Videos de Entrenamiento (Fight): {len(list(TRAIN_FIGHT_DIR.glob('*.*')))}")
print(f"Videos de Entrenamiento (No-Fight): {len(list(TRAIN_NOFIGHT_DIR.glob('*.*')))}")
print(f"Videos de Validación (Fight): {len(list(VAL_FIGHT_DIR.glob('*.*')))}")
print(f"Videos de Validación (No-Fight): {len(list(VAL_NOFIGHT_DIR.glob('*.*')))}")

In [ ]:
import cv2
import numpy as np
from ultralytics import YOLO

# Change this path to your trained YOLO weights
YOLO_WEIGHTS = "/content/runs/detect/train/weights/best.pt"
yolo = YOLO(YOLO_WEIGHTS)

# For your 2-class detector:
CLS_VIOLENCE = 0
CLS_WEAPON   = 1

def frame_features_from_yolo(result, img_w, img_h):
    v_count = v_sumconf = v_area = v_maxconf = 0.0
    w_count = w_sumconf = w_area = w_maxconf = 0.0

    boxes = result.boxes
    if boxes is None or len(boxes) == 0:
        return np.array([0,0,0,0, 0,0,0,0], dtype=np.float32)

    cls = boxes.cls.cpu().numpy().astype(int)
    conf = boxes.conf.cpu().numpy().astype(float)
    xyxy = boxes.xyxy.cpu().numpy().astype(float)

    for c, p, (x1,y1,x2,y2) in zip(cls, conf, xyxy):
        area = max(0.0, (x2-x1)) * max(0.0, (y2-y1)) / (img_w * img_h + 1e-9)
        if c == CLS_VIOLENCE:
            v_count += 1; v_sumconf += p; v_area += area; v_maxconf = max(v_maxconf, p)
        elif c == CLS_WEAPON:
            w_count += 1; w_sumconf += p; w_area += area; w_maxconf = max(w_maxconf, p)

    return np.array([v_count, v_sumconf, v_area, v_maxconf,
                     w_count, w_sumconf, w_area, w_maxconf], dtype=np.float32)

def extract_sequence_features(video_path, fps_target=10, max_seconds=None):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise ValueError(f"Cannot open video: {video_path}")

    src_fps = cap.get(cv2.CAP_PROP_FPS) or 30
    step = max(1, int(round(src_fps / fps_target)))

    feats = []
    idx = 0
    kept = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        if idx % step != 0:
            idx += 1
            continue

        h, w = frame.shape[:2]
        res = yolo(frame, verbose=False)[0]
        feats.append(frame_features_from_yolo(res, w, h))

        kept += 1
        if max_seconds is not None and (kept / fps_target) >= max_seconds:
            break

        idx += 1

    cap.release()
    if len(feats) == 0:
        return np.zeros((0, 8), dtype=np.float32)
    return np.stack(feats, axis=0)

In [ ]:
from pathlib import Path
from tqdm import tqdm
import numpy as np

def make_windows(features, window=32, stride=8):
    X = []
    T = features.shape[0]
    if T == 0:
        return np.zeros((0, window, features.shape[1]), dtype=np.float32)
    for start in range(0, max(1, T - window + 1), stride):
        clip = features[start:start+window]
        if clip.shape[0] < window:
            pad = np.zeros((window - clip.shape[0], clip.shape[1]), dtype=np.float32)
            clip = np.vstack([clip, pad])
        X.append(clip)
    return np.stack(X, axis=0)

def build_sequence_dataset(videos_root, window=32, stride=8, fps_target=10, max_seconds=8):
    root = Path(videos_root)
    X_all, y_all = [], []
    classes = [("no_fight", 0), ("fight", 1)]

    for cname, y in classes:
        folder = root / cname

        # Validación: Revisar si la carpeta existe antes de buscar videos
        if not folder.exists():
            print(f"Advertencia: La carpeta {folder} no existe.")
            continue

        vids = sorted([p for p in folder.glob("*") if p.suffix.lower() in [".mp4", ".avi", ".mov", ".mkv"]])

        for vp in tqdm(vids, desc=f"Extracting {cname}"):
            feats = extract_sequence_features(str(vp), fps_target=fps_target, max_seconds=max_seconds)
            Xw = make_windows(feats, window=window, stride=stride)
            if Xw.shape[0] == 0:
                continue
            X_all.append(Xw)
            y_all.append(np.full((Xw.shape[0],), y, dtype=np.int64))

    # Validación: Prevenir el error de concatenación si las listas están vacías
    if len(X_all) == 0:
        print(f"No se encontraron videos válidos para procesar en {root}")
        return np.array([]), np.array([])

    X = np.concatenate(X_all, axis=0)
    y = np.concatenate(y_all, axis=0)
    return X, y

# ==========================================
# EJECUCIÓN CORREGIDA PARA TRAIN Y VAL
# ==========================================

# 1. Procesar Entrenamiento
print("Procesando videos de Entrenamiento (Train)...")
X_train, y_train = build_sequence_dataset(OUT_VIDEOS / "train", window=32, stride=8, fps_target=10, max_seconds=8)

if len(y_train) > 0:
    print("Train X:", X_train.shape, "Train y:", y_train.shape, "pos_rate:", y_train.mean())

print("-" * 40)

# 2. Procesar Validación
print("Procesando videos de Validación (Val)...")
X_val, y_val = build_sequence_dataset(OUT_VIDEOS / "val", window=32, stride=8, fps_target=10, max_seconds=8)

if len(y_val) > 0:
    print("Val X:", X_val.shape, "Val y:", y_val.shape, "pos_rate:", y_val.mean())

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import classification_report
import numpy as np

class SeqDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

class LSTMClassifier(nn.Module):
    def __init__(self, feat_dim, hidden=128, dropout=0.3):
        super().__init__()
        self.lstm = nn.LSTM(feat_dim, hidden, batch_first=True, bidirectional=True)
        self.head = nn.Sequential(
            nn.Linear(hidden*2, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 2)
        )

    def forward(self, x):
        out, _ = self.lstm(x)
        last = out[:, -1, :]
        return self.head(last)

class StrongLSTMClassifier(nn.Module):
    def __init__(self, feat_dim, hidden=256, num_layers=2, dropout=0.4):
        super().__init__()

        # 1. Un LSTM más profundo y ancho
        self.lstm = nn.LSTM(
            input_size=feat_dim,
            hidden_size=hidden,
            num_layers=num_layers, # Ahora tiene 2 capas por defecto
            batch_first=True,
            bidirectional=True,
            dropout=dropout if num_layers > 1 else 0 # Dropout entre las capas del LSTM
        )

        # 2. Una cabeza clasificadora más robusta
        self.head = nn.Sequential(
            nn.Linear(hidden * 2, 256),
            nn.BatchNorm1d(256),  # Ayuda a que la red converja más rápido y mejor
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(256, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(dropout / 2),

            nn.Linear(64, 2)
        )

    def forward(self, x):
        out, _ = self.lstm(x)
        # Tomamos la salida del último paso temporal
        last = out[:, -1, :]
        return self.head(last)

import torch.nn.functional as F

class TemporalAttention(nn.Module):
    def __init__(self, hidden_size):
        super().__init__()
        # Capa para calcular la importancia de cada frame
        self.attention = nn.Linear(hidden_size * 2, 1) # *2 porque es bidireccional

    def forward(self, lstm_out):
        # lstm_out: [batch_size, seq_len, hidden_size * 2]

        # Calculamos los pesos de atención para cada frame
        attn_weights = F.softmax(self.attention(lstm_out), dim=1) # [batch_size, seq_len, 1]

        # Multiplicamos la salida del LSTM por sus pesos y sumamos
        context = torch.sum(attn_weights * lstm_out, dim=1) # [batch_size, hidden_size * 2]
        return context, attn_weights

class UltimateLSTMClassifier(nn.Module):
    def __init__(self, feat_dim, hidden=256, num_layers=3, dropout=0.5):
        super().__init__()

        # 1. LSTM Extra Profundo (3 capas de memoria)
        self.lstm = nn.LSTM(
            input_size=feat_dim,
            hidden_size=hidden,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout
        )

        # 2. Nuestro nuevo "Cerebro" de Atención
        self.attention = TemporalAttention(hidden)

        # 3. Cabeza Clasificadora de Alto Rendimiento
        self.head = nn.Sequential(
            nn.Linear(hidden * 2, 512),
            nn.LayerNorm(512),  # LayerNorm es más estable que BatchNorm para secuencias
            nn.GELU(),          # GELU es una activación más moderna y suave que ReLU
            nn.Dropout(dropout),

            nn.Linear(512, 128),
            nn.LayerNorm(128),
            nn.GELU(),
            nn.Dropout(dropout / 2),

            nn.Linear(128, 2)
        )

    def forward(self, x):
        # Pasamos por el LSTM
        out, _ = self.lstm(x)

        # En lugar de tomar el último frame, usamos Atención para extraer lo mejor de toda la secuencia
        context, _ = self.attention(out)

        # Clasificamos
        return self.head(context)

# Usamos nuestras nuevas variables X_train, y_train, X_val, y_val
train_loader = DataLoader(SeqDataset(X_train, y_train), batch_size=64, shuffle=True)
val_loader   = DataLoader(SeqDataset(X_val, y_val), batch_size=64, shuffle=False)

device = "cuda" if torch.cuda.is_available() else "cpu"

# Usamos X_train en lugar de X
model = UltimateLSTMClassifier(feat_dim=X_train.shape[2]).to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
crit = nn.CrossEntropyLoss()

def eval_model():
    model.eval()
    ys, ps = [], []
    with torch.no_grad():
        for xb, yb in val_loader:
            xb = xb.to(device)
            logits = model(xb)
            pred = logits.argmax(1).cpu().numpy()
            ys.append(yb.numpy())
            ps.append(pred)
    ys = np.concatenate(ys); ps = np.concatenate(ps)
    print(classification_report(ys, ps, target_names=["no_fight", "fight"]))

for epoch in range(1, 25):
    model.train()
    total = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        opt.zero_grad()
        loss = crit(model(xb), yb)
        loss.backward()
        opt.step()
        total += loss.item()
    print(f"Epoch {epoch} | loss={total/len(train_loader):.4f}")
    eval_model()

torch.save(model.state_dict(), "/content/lstm_fight.pt")
print("Saved: /content/lstm_fight.pt")

#Inference


In [ ]:
import cv2
import numpy as np
import torch
import torch.nn as nn
from ultralytics import YOLO
from collections import deque

# -------- Load models --------
YOLO_WEIGHTS = "/content/runs/detect/train/weights/best.pt"   # change if needed
LSTM_WEIGHTS = "/content/lstm_fight.pt"                      # change if needed

yolo = YOLO(YOLO_WEIGHTS)

# Your YOLO class names (ensure order matches training)
YOLO_NAMES = ["violence", "weapon"]  # class 0, class 1

# LSTM model definition (must match training)
import torch.nn.functional as F

class TemporalAttention(nn.Module):
    def __init__(self, hidden_size):
        super().__init__()
        # Capa para calcular la importancia de cada frame
        self.attention = nn.Linear(hidden_size * 2, 1) # *2 porque es bidireccional

    def forward(self, lstm_out):
        # lstm_out: [batch_size, seq_len, hidden_size * 2]

        # Calculamos los pesos de atención para cada frame
        attn_weights = F.softmax(self.attention(lstm_out), dim=1) # [batch_size, seq_len, 1]

        # Multiplicamos la salida del LSTM por sus pesos y sumamos
        context = torch.sum(attn_weights * lstm_out, dim=1) # [batch_size, hidden_size * 2]
        return context, attn_weights

class UltimateLSTMClassifier(nn.Module):
    def __init__(self, feat_dim, hidden=256, num_layers=3, dropout=0.5):
        super().__init__()

        # 1. LSTM Extra Profundo (3 capas de memoria)
        self.lstm = nn.LSTM(
            input_size=feat_dim,
            hidden_size=hidden,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout
        )

        # 2. Nuestro nuevo "Cerebro" de Atención
        self.attention = TemporalAttention(hidden)

        # 3. Cabeza Clasificadora de Alto Rendimiento
        self.head = nn.Sequential(
            nn.Linear(hidden * 2, 512),
            nn.LayerNorm(512),  # LayerNorm es más estable que BatchNorm para secuencias
            nn.GELU(),          # GELU es una activación más moderna y suave que ReLU
            nn.Dropout(dropout),

            nn.Linear(512, 128),
            nn.LayerNorm(128),
            nn.GELU(),
            nn.Dropout(dropout / 2),

            nn.Linear(128, 2)
        )

    def forward(self, x):
        # Pasamos por el LSTM
        out, _ = self.lstm(x)

        # En lugar de tomar el último frame, usamos Atención para extraer lo mejor de toda la secuencia
        context, _ = self.attention(out)

        # Clasificamos
        return self.head(context)

device = "cuda" if torch.cuda.is_available() else "cpu"
lstm = UltimateLSTMClassifier(feat_dim=8).to(device)
lstm.load_state_dict(torch.load(LSTM_WEIGHTS, map_location=device))
lstm.eval()

def frame_features_from_yolo(result, img_w, img_h):
    """
    Build an 8D feature vector from YOLO detections for temporal classification.
    """
    # [viol_count, viol_sumconf, viol_area, viol_maxconf, weap_count, weap_sumconf, weap_area, weap_maxconf]
    v_count = v_sumconf = v_area = v_maxconf = 0.0
    w_count = w_sumconf = w_area = w_maxconf = 0.0

    boxes = result.boxes
    if boxes is None or len(boxes) == 0:
        return np.array([0,0,0,0, 0,0,0,0], dtype=np.float32)

    cls = boxes.cls.cpu().numpy().astype(int)
    conf = boxes.conf.cpu().numpy().astype(float)
    xyxy = boxes.xyxy.cpu().numpy().astype(float)

    for c, p, (x1,y1,x2,y2) in zip(cls, conf, xyxy):
        area = max(0.0, (x2-x1)) * max(0.0, (y2-y1)) / (img_w * img_h + 1e-9)
        if c == 0:  # violence
            v_count += 1; v_sumconf += p; v_area += area; v_maxconf = max(v_maxconf, p)
        elif c == 1:  # weapon
            w_count += 1; w_sumconf += p; w_area += area; w_maxconf = max(w_maxconf, p)

    return np.array([v_count, v_sumconf, v_area, v_maxconf,
                     w_count, w_sumconf, w_area, w_maxconf], dtype=np.float32)

def lstm_label_from_window(window_feats):
    """
    window_feats: np.array [T, 8]
    Returns a hard label: "FIGHT" or "NO FIGHT"
    """
    xb = torch.tensor(window_feats[None, ...], dtype=torch.float32).to(device)
    with torch.no_grad():
        logits = lstm(xb)
        pred = int(torch.argmax(logits, dim=1).item())
    return "FIGHT" if pred == 1 else "NO FIGHT"

def draw_detections(frame, result, topk=20):
    """
    Draw YOLO detections on a frame.
    """
    boxes = result.boxes
    if boxes is None or len(boxes) == 0:
        return frame

    cls = boxes.cls.cpu().numpy().astype(int)
    conf = boxes.conf.cpu().numpy().astype(float)
    xyxy = boxes.xyxy.cpu().numpy().astype(int)

    # Sort by confidence
    order = np.argsort(-conf)[:topk]

    for i in order:
        x1, y1, x2, y2 = xyxy[i]
        c = cls[i]
        p = conf[i]
        label = f"{YOLO_NAMES[c]} {p:.2f}"

        cv2.rectangle(frame, (x1,y1), (x2,y2), (0,255,0), 2)
        cv2.putText(frame, label, (x1, max(0,y1-6)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,255,0), 2, cv2.LINE_AA)
    return frame

In [ ]:
def annotate_video_with_yolo_lstm(
    input_video,
    output_video="/content/output_annotated.mp4",
    window=32,
    fps_target=None,          # None = keep original fps
    conf_thres=0.65,
    max_frames=None
):
    cap = cv2.VideoCapture(input_video)
    if not cap.isOpened():
        raise ValueError(f"Cannot open video: {input_video}")

    src_fps = cap.get(cv2.CAP_PROP_FPS) or 30
    W = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    H = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    # Optional downsample to fps_target
    if fps_target is not None:
        step = max(1, int(round(src_fps / fps_target)))
        out_fps = fps_target
    else:
        step = 1
        out_fps = src_fps

    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    writer = cv2.VideoWriter(output_video, fourcc, out_fps, (W, H))

    feat_queue = deque(maxlen=window)
    frame_idx = 0
    kept = 0
    current_label = "NORMAL (Sin Pelea)"

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        if frame_idx % step != 0:
            frame_idx += 1
            continue

        # YOLO inference
        res = yolo(frame, conf=conf_thres, verbose=False)[0]

        # Update temporal features
        feat_queue.append(frame_features_from_yolo(res, W, H))

        # Once window is full, update label
        if len(feat_queue) == window:
            window_feats = np.stack(list(feat_queue), axis=0)

            # 1. El LSTM nos dice si hay pelea o no
            lstm_pred = lstm_label_from_window(window_feats)

            # 2. NUEVA LÓGICA: Preguntamos a YOLO si vio un arma
            if lstm_pred == "FIGHT":
                # window_feats[:, 4] es la columna w_count (conteo de armas)
                total_armas = np.sum(window_feats[:, 4])

                if total_armas > 0:
                    current_label = "PELEA (Con Arma)"
                else:
                    current_label = "PELEA (Agresion Fisica)"
            else:
                current_label = "NORMAL (Sin Pelea)"

        # Draw boxes + global label
        out_frame = frame.copy()
        out_frame = draw_detections(out_frame, res)

        # Elegir color del texto: Rojo para pelea, Verde para normal
        # (Los colores en OpenCV son BGR, no RGB)
        color_texto = (0, 0, 255) if "PELEA" in current_label else (0, 255, 0)

        # Put global label on the frame (borde blanco para que resalte)
        cv2.putText(out_frame, f"ESTADO: {current_label}", (15, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 4, cv2.LINE_AA)
        cv2.putText(out_frame, f"ESTADO: {current_label}", (15, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, color_texto, 2, cv2.LINE_AA)

        writer.write(out_frame)
        kept += 1
        frame_idx += 1

        if max_frames is not None and kept >= max_frames:
            break

    cap.release()
    writer.release()
    return output_video

# --- EJECUCIÓN ---
out_path = annotate_video_with_yolo_lstm("/content/dataset_videos/Real Life Violence Dataset/Violence/V_999.mp4")
print("¡Listo! Video guardado en:", out_path)